# Lab Assignment 2

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA = '/content/drive/MyDrive/Images/'
except ImportError:
    DATA = ''

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score)

Mounted at /content/drive


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

spam = pd.read_csv(DATA + 'spam.csv', encoding='latin-1').iloc[:, :2]
spam.columns = ['Labels', 'SMS']
spam['Labels'] = spam['Labels'].map({'spam': 1, 'ham': 0})
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    spam['SMS'].values, spam['Labels'].values, test_size=0.2, random_state=50)

results = {}
for name, vec in [('CountVectorizer', CountVectorizer(stop_words='english')),
                  ('TF-IDF', TfidfVectorizer(stop_words='english'))]:
    Xtr = vec.fit_transform(Xs_train)
    Xte = vec.transform(Xs_test)
    model = MultinomialNB().fit(Xtr, ys_train)
    pred = model.predict(Xte)
    results[name] = {
        'Train Acc': accuracy_score(ys_train, model.predict(Xtr)),
        'Test Acc': accuracy_score(ys_test, pred),
    }
    print(f"===== {name} (stop_words='english') | features: {Xtr.shape[1]} =====")
    print(confusion_matrix(ys_test, pred))
    print(classification_report(ys_test, pred, target_names=['ham', 'spam']))

from sklearn.metrics import precision_score, recall_score, f1_score
for name, vec in [('CountVectorizer', CountVectorizer(stop_words='english')),
                  ('TF-IDF', TfidfVectorizer(stop_words='english'))]:
    Xtr = vec.fit_transform(Xs_train); Xte = vec.transform(Xs_test)
    pred = MultinomialNB().fit(Xtr, ys_train).predict(Xte)
    results[name].update({'Spam Precision': precision_score(ys_test, pred),
                          'Spam Recall': recall_score(ys_test, pred),
                          'Spam F1': f1_score(ys_test, pred)})

comparison = pd.DataFrame(results).T
display(comparison)

===== CountVectorizer (stop_words='english') | features: 7466 =====
[[951   3]
 [ 16 145]]
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       954
        spam       0.98      0.90      0.94       161

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115

===== TF-IDF (stop_words='english') | features: 7466 =====
[[954   0]
 [ 44 117]]
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       954
        spam       1.00      0.73      0.84       161

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



,Train Acc,Test Acc,Spam Precision,Spam Recall,Spam F1
CountVectorizer,0.994615,0.982960,0.97973,0.900621,0.938511
TF-IDF,0.984294,0.960538,1.00000,0.726708,0.841727


In [4]:
best = comparison['Test Acc'].idxmax()
print(f"Conclusion: by test accuracy, '{best}' is the better feature representation for spam.csv.")
print("Spam F1 -> " + ", ".join(f"{n}: {r['Spam F1']:.4f}" for n, r in results.items()))
print("Spam recall -> " + ", ".join(f"{n}: {r['Spam Recall']:.4f}" for n, r in results.items()))

Conclusion: by test accuracy, 'CountVectorizer' is the better feature representation for spam.csv.
Spam F1 -> CountVectorizer: 0.9385, TF-IDF: 0.8417
Spam recall -> CountVectorizer: 0.9006, TF-IDF: 0.7267
